In [ ]:
import json
import time
from datetime import datetime
from pathlib import Path
from conexion_host import ConexionHost
from JUEGO_BODEGA import juego_bodega
from simondice import ejecutar_minijuego_simon

BASE_DIR = Path(__file__).resolve().parent
CONFIG_PATH = BASE_DIR / "config_grupo.json"

# Cambiar este valor para solicitar otra etapa al host de pruebas.
ETAPA_PRUEBA = "R1"


def cargar_config():
    return json.loads(CONFIG_PATH.read_text(encoding="utf-8"))


def probar_ronda_1():
    config = cargar_config()
    conexion = ConexionHost(config)
    player_id = config.get("player_id", "P01")

    try:
        print("Conectando al host de pruebas...")
        conexion.conectar()

        # 1. Registro inicial en el Lobby (Join -> Accepted)
        event_id_join = f"{player_id}-00001"
        msg_join = {
            "timestamp": datetime.now().isoformat(),
            "player_id": player_id,
            "Action": "Join",
            "EventID": event_id_join,
            "Stage": "Lobby"
        }
        conexion.enviar_json(msg_join)
        print("Mensaje Join enviado. Esperando confirmación Accepted...")

        aceptado = False
        stage_start_guardado = None  # Aquí guardaremos el StageStart si llega junto con el Accepted

        print("Esperando confirmación Accepted...")
        while not aceptado:
            mensajes = conexion.leer_nuevos_json()
            
            for msg in mensajes:
                tipo_accion = msg.get("Action")
                
                # Verificamos si es el Accepted de nuestro Join
                if tipo_accion == "Accepted" and msg.get("ReplyTo") == event_id_join:
                    print("¡Ingreso al Lobby confirmado exitosamente!")
                    aceptado = True
                    
                # Si de pasada viene el StageStart, lo guardamos para no perderlo
                elif tipo_accion == "StageStart" and msg.get("Stage") == ETAPA_PRUEBA:
                    print(f"¡StageStart detectado de forma anticipada para {ETAPA_PRUEBA}!")
                    stage_start_guardado = msg
                    
            if not aceptado:
                time.sleep(0.2)

        # 2. Verificar o esperar el StageStart
        if stage_start_guardado:
            print(f"Usando el StageStart que ya teníamos guardado para {ETAPA_PRUEBA}...")
        else:
            print(f"Esperando StageStart para la etapa {ETAPA_PRUEBA}...")
            while not stage_start_guardado:
                mensajes = conexion.leer_nuevos_json()
                for msg in mensajes:
                    print("DEBUG - Mensajes recibidos en la bandeja:", mensajes)
                    if msg.get("Action") == "StageStart" and msg.get("Stage") == ETAPA_PRUEBA:
                        stage_start_guardado = msg
                        print(f"¡StageStart recibido!")
                        break
                if not stage_start_guardado:
                    time.sleep(0.2)

    
        # 3. Enviar StageReady y esperar la activación[cite: 7]
        event_id_ready = f"{player_id}-ready-r1"
        msg_ready = {
            "timestamp": datetime.now().isoformat(),
            "player_id": player_id,
            "Action": "StageReady",
            "EventID": event_id_ready,
            "Stage": ETAPA_PRUEBA
        }
        conexion.enviar_json(msg_ready)
        print("Mensaje StageReady enviado. Esperando StageActive...")

        stage_activo = False
        while not stage_activo:
            mensajes = conexion.leer_nuevos_json()
            for msg in mensajes:
                if msg.get("Action") == "StageActive" and msg.get("Stage") == ETAPA_PRUEBA:
                    print(f"¡La etapa {ETAPA_PRUEBA} está oficialmente ACTIVA!")
                    stage_activo = True
                    break
            if not stage_activo:
                time.sleep(0.2)

        # 4. Ciclo continuo de minijuegos (Assign -> Result -> ResultAccepted)[cite: 7]
        print("Iniciando ciclo de recepción de minijuegos de la Ronda 1...")
        ciclo_activo = True
        intento_contador = 0

        while ciclo_activo:
            mensajes = conexion.leer_nuevos_json()
            for msg in mensajes:
                accion = msg.get("Action")
                
                # Si recibimos una asignación de minijuego
                if accion == "Assign" and msg.get("Stage") == ETAPA_PRUEBA:
                    game_id = msg.get("GameID")
                    attempt = msg.get("Attempt")
                    print(f"\n[ASIGNACIÓN] Minijuego ID: {game_id}, Intento: {attempt}")

                    # Seleccionar el minijuego según el GameID enviado por el servidor
                    if game_id == 1:
                        print("Iniciando: Juego Bodega...")
                        respuesta = juego_bodega()
                    elif game_id == 2:
                        print("Iniciando: Simón Dice...")
                        respuesta = ejecutar_minijuego_simon(max_time=15)
                    else:
                        print(f"GameID {game_id} desconocido.")
                        respuesta = {"Result": "Lose", "Score": 0}

                    resultado = respuesta["Result"]
                    score = respuesta["Score"]
                    print(f"Minijuego finalizado -> Resultado: {resultado}, Puntaje: {score}")

                    # Enviar el mensaje Result al servidor
                    intento_contador += 1
                    event_id_res = f"{player_id}-res-{intento_contador}"
                    msg_result = {
                        "timestamp": datetime.now().isoformat(),
                        "player_id": player_id,
                        "Action": "Result",
                        "EventID": event_id_res,
                        "Stage": ETAPA_PRUEBA,
                        "GameID": game_id,
                        "Attempt": attempt,
                        "Result": resultado,
                        "Score": score
                    }
                    conexion.enviar_json(msg_result)
                    print(f"Mensaje Result enviado (EventID: {event_id_res})")
                    

                    # Esperar la confirmación ResultAccepted[cite: 7]
                    res_aceptado = False
                    while not res_aceptado:
                        sub_mensajes = conexion.leer_nuevos_json()
                        for sm in sub_mensajes:
                            if sm.get("Action") == "ResultAccepted" and sm.get("ReplyTo") == event_id_res:
                                print("¡ResultAccepted recibido! Intercambio de minijuego finalizado.")
                                res_aceptado = True
                                break
                        if not res_aceptado:
                            time.sleep(0.2)

                elif accion in ["StageStart", "FinalStart", "GameOver"]:
                    print(f"Se detectó un cambio de etapa o fin de partida ({accion}). Saliendo de la prueba.")
                    ciclo_activo = False
                    break

            if ciclo_activo:
                time.sleep(0.2)

    except Exception as e:
        print(f"Error durante la prueba de R1: {e}")
    finally:
        conexion.cerrar_conexion()
        print("Conexión cerrada de forma ordenada.")


if __name__ == "__main__":
    probar_ronda_1()